# Paliers M2 et M3 — faire entrer le texte des libellés

**Rappel de l'échelle** (détail complet dans [`baseline.ipynb`](baseline.ipynb)) : M0 = déclaratif. M1 = + comportement du compte, *sans lire le sens* des libellés — conclusion de `baseline.ipynb` : aucun gain statistiquement significatif, le signal utile est ailleurs. Ici, on ajoute la **compréhension du texte** :

- **M2 — règles.** Un lexique de *racines* (stems) robustes au bruit retrouve la catégorie de chaque transaction (salaire, momo, njangui, agios…) malgré abréviations, troncatures et coquilles. On en déduit la *part de chaque catégorie* par client.
- **M3 — représentation vectorielle.** On vectorise les libellés par TF-IDF de n-grammes de **caractères** (robuste aux fautes, hors-ligne) réduits par SVD. C'est l'alternative « embeddings » qui ne dépend pas d'un lexique écrit à la main.

Même méthode que dans `baseline.ipynb` : pour chaque palier on **cherche le meilleur `C`** par validation croisée, on regarde **la table de coefficients**, et on compare statistiquement au palier précédent (test de DeLong). Le code générique (pipeline, recherche de `C`, coefficients, métriques, DeLong, variables comportementales, audit d'équité) vit dans [`scoring_utils.py`](scoring_utils.py), et la catégorisation par règles dans [`categorisation.py`](categorisation.py) — rien n'est redéfini ici.

**Consigne permanente : audit d'équité à chaque palier** (ratio d'approbation par secteur *et* par sexe, et part d'injustice via la vérité-terrain `gt_defaut_true`). La correction de ce biais est traitée à part, dans [`equite.ipynb`](equite.ipynb).

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

from categorisation import categoriser_libelle, construire_parts_categories
from scoring_utils import (
    audit_equite,
    chaine_delong,
    construire_pipeline,
    construire_variables_comportementales,
    delong_test,
    evaluer,
    extraire_coefficients,
    rechercher_meilleur_C,
)

SEED = 42
TARGET = "defaut_90j"

DECLARATIF_NUM = ["age", "revenu_declare", "anciennete_mois"]
DECLARATIF_CAT = ["zone", "secteur", "region"]
COMPORTEMENTAL_NUM = [
    "nb_tx", "pct_debits", "inflow", "outflow", "net_flow",
    "mean_abs", "std_abs", "max_abs", "cv_abs",
    "nb_jours_actifs", "tx_par_jour", "ecart_revenu",
]

# même grille que dans baseline.ipynb, pour rester comparable d'un notebook à l'autre
GRILLE_C = [0.001, 0.01, 0.1, 1, 10, 100]

## Étape 1 — Charger les données

Mêmes fichiers que `baseline.ipynb` : `clients_synth.csv` et `transactions_synth.csv`.

In [2]:
clients = pd.read_csv("clients_synth.csv")
tx = pd.read_csv("transactions_synth.csv")

print(f"clients : {clients.shape[0]} lignes | transactions : {tx.shape[0]} lignes")
tx[["libelle", "gt_categorie", "gt_mensonge"]].head()

clients : 2000 lignes | transactions : 91666 lignes


,libelle,gt_categorie,gt_mensonge
0,paiMt-FAct.EnEo/REF9731984,ELEC,0
1,paIEE LoYErBlR REF4806837 AGMAR 1/04,LOYER,0
2,eNVoIE Momo au boSs/TPE1761/25/08,MOMO,0
3,OpR nJAN,NJANGUI,0
4,DeBcotisAtION_TontInE REF383522 TPE50248,NJANGUI,0


## Rappel — variables comportementales (M1)

`construire_variables_comportementales` (importée de `scoring_utils.py`, détaillée variable par variable dans `baseline.ipynb`) résume l'historique de transactions en une ligne par client — volume, flux, régularité — **sans lire le sens** des libellés. M2 et M3 partent de cette même base et y ajoutent la compréhension du texte.

In [3]:
comp = construire_variables_comportementales(tx, clients)
comp.head()

,client_id,nb_tx,pct_debits,inflow,outflow,mean_abs,std_abs,max_abs,nb_jours_actifs,net_flow,cv_abs,tx_par_jour,ecart_revenu
0,1,28,0.821429,1970486.0,1958456.0,140319.357143,352120.284243,1481199.0,26,12030.0,2.509421,1.076923,-5.025951
1,2,58,0.810345,4128443.0,2782051.0,119146.448276,251905.005755,1009317.0,44,1346392.0,2.114247,1.318182,-6.908895
2,3,36,0.750000,2775876.0,2040831.0,133797.416667,292261.844375,1364223.0,25,735045.0,2.184361,1.440000,-3.895725
3,4,21,0.857143,179598.0,1871038.0,97649.333333,319716.352365,1489911.0,19,-1691440.0,3.274127,1.105263,-0.814121
4,5,29,0.931034,397596.0,4884317.0,182134.931034,357177.292957,1209253.0,20,-4486721.0,1.961059,1.450000,0.181901


## Étape 2 — Palier M2 : comprendre le texte par des règles

Les libellés sont sales : majuscules anarchiques, séparateurs mangés, codes d'agence/terminal, références numériques, accents perdus, coquilles (`paiMt-FAct.EnEo/REF9731984`, `DeBcotisAtION_TontInE REF383522 TPE50248`, ...). L'approche par règles, implémentée dans [`categorisation.py`](categorisation.py), se fait en trois temps :

1. un **lexique** de racines courtes par catégorie (`NJANG`, `CAMWAT`, `DECOUV`, `BENSK`, ...), choisies pour survivre aux abréviations et troncatures ;
2. une fonction de **normalisation** qui nettoie le libellé (accents, chiffres, ponctuation) avant de chercher ces racines ;
3. une fonction de **catégorisation** qui retient la catégorie dont le plus grand nombre de racines apparaît dans le libellé normalisé.

Exemple sur un libellé réel du jeu de données :

In [4]:
categoriser_libelle("paiMt-FAct.EnEo/REF9731984")

'ELEC'

**De la catégorie par transaction à une variable par client.** Une fois chaque transaction catégorisée, on calcule pour chaque client la *part* de ses transactions dans chaque catégorie (`PART_MOMO`, `PART_AGIOS`, ...) — ce sont ces parts qui deviendront des variables du modèle M2. On en profite pour vérifier la qualité de la catégorisation contre `gt_categorie` (la vraie catégorie, connue seulement parce que les données sont synthétiques) : `gt_mensonge` marque les libellés volontairement déguisés (un pari maquillé en "ACHAT DIVERS"), que les règles ne peuvent pas voir — on regarde donc la précision globale et hors-mensonge séparément.

In [5]:
parts, tx_categorise = construire_parts_categories(tx)
PART_COLS = [c for c in parts.columns if c.startswith("PART_")]

precision = (tx_categorise.cat_regle == tx_categorise.gt_categorie).mean()
hors_mensonge = tx_categorise[tx_categorise.gt_mensonge == 0]
precision_hors_mensonge = (hors_mensonge.cat_regle == hors_mensonge.gt_categorie).mean()

print(f"précision de la catégorisation par règles : {precision:.3f} global | "
      f"{precision_hors_mensonge:.3f} hors mensonge (les règles ne voient pas le mensonge)")
parts.head()

précision de la catégorisation par règles : 0.721 global | 0.777 hors mensonge (les règles ne voient pas le mensonge)


,client_id,PART_AGIOS,PART_CHOP,PART_DASH,PART_EAU,PART_ELEC,PART_INCONNU,PART_LOYER,PART_MOMO,PART_NJANGUI,PART_PARI,PART_SALAIRE,PART_TRANSPORT
0,1,0.035714,0.178571,0.035714,0.000000,0.107143,0.178571,0.035714,0.178571,0.035714,0.071429,0.071429,0.071429
1,2,0.000000,0.155172,0.017241,0.017241,0.068966,0.137931,0.068966,0.120690,0.000000,0.103448,0.155172,0.155172
2,3,0.055556,0.055556,0.111111,0.000000,0.083333,0.194444,0.027778,0.138889,0.000000,0.083333,0.166667,0.083333
3,4,0.000000,0.238095,0.047619,0.047619,0.000000,0.142857,0.047619,0.095238,0.000000,0.047619,0.190476,0.142857
4,5,0.172414,0.034483,0.000000,0.000000,0.068966,0.172414,0.206897,0.137931,0.034483,0.034483,0.068966,0.068966


## Étape 3 — Palier M3 : représentation vectorielle du texte

Le lexique M2 dépend d'une liste de racines écrites à la main : il rate tout mot qui n'y figure pas. L'alternative « embeddings » évite ce biais : on vectorise le texte, sans dictionnaire, et on laisse la régression logistique trouver ce qui compte.

- **N-grammes de caractères plutôt que de mots** — `TfidfVectorizer(analyzer="char_wb", ngram_range=(3,5))` : robuste aux fautes, abréviations et troncatures (ce que M2 gère à la main, ceci le gère statistiquement), et fonctionne hors-ligne, sans modèle pré-entraîné à télécharger.
- **SVD à 30 dimensions** — les milliers de n-grammes possibles sont réduits à 30 combinaisons denses, pour rester dans un ordre de grandeur raisonnable face à ~1600 clients d'entraînement (`TruncatedSVD(n_components=30)`).

C'est la même construction que dans `construire_pipeline(..., texte_col=...)` de `scoring_utils.py` — la seule chose à faire ici est de fournir un texte par client : tous ses libellés concaténés.

In [6]:
texte_par_client = (tx.groupby("client_id").libelle
                    .apply(lambda s: " ".join(map(str, s)))
                    .rename("texte")
                    .reset_index())
texte_par_client.head()

,client_id,texte
0,1,paiMt-FAct.EnEo/REF9731984 paIEE LoYErBlR REF...
1,2,dEPoTREF5790709AGNGD pmTcAmwAtER/REF347874XAGD...
2,3,vIR.oR-MOnEnvoi REF3269837 AGBER TPE14315 659*...
3,4,Opr/DAssh AvaNtsAlr/REF2775832/690****34/06/03...
4,5,Deb.paRiFOoT*TPE97315*6729***67 ret gabOM-casH...


## Étape 4 — Construire X/y et le split train/test

On fusionne clients + comportement (M1) + parts de catégories (M2) + texte (M3), on retire les `gt_*`, puis on découpe en train/test avec le **même `test_size` et le même `random_state`** que `baseline.ipynb` — ce qui place les mêmes clients en test dans les deux notebooks, et rend les AUC de M0 à M3 directement comparables entre eux.

In [7]:
df = (clients
      .merge(comp, on="client_id", how="left")
      .merge(parts, on="client_id", how="left")
      .merge(texte_par_client, on="client_id", how="left"))
df[COMPORTEMENTAL_NUM + PART_COLS] = df[COMPORTEMENTAL_NUM + PART_COLS].fillna(0)
df["texte"] = df["texte"].fillna("")

y = df[TARGET].values
colonnes_gt = [c for c in df.columns if c.startswith("gt_")]
X = df.drop(columns=colonnes_gt + [TARGET, "client_id"])

Xtr, Xte, ytr, yte = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=SEED
)
dfte = df.loc[Xte.index]  # pour l'audit d'équité, plus loin

print(f"train {len(Xtr)} | test {len(Xte)}")
print(f"taux de défaut : train {ytr.mean():.3f} | test {yte.mean():.3f}")

train 1600 | test 400
taux de défaut : train 0.159 | test 0.158


## Étape 5 — Rappel rapide M0 et M1

Juste ce qu'il faut pour amorcer la chaîne de comparaison M1→M2→M3 : recherche de `C`, évaluation sur le test. Le détail variable par variable, les tables de coefficients et la comparaison à `C` fixe sont dans `baseline.ipynb` — pas repris ici.

In [8]:
pipeline_m0 = construire_pipeline(DECLARATIF_NUM, DECLARATIF_CAT, seed=SEED)
modele_m0, _ = rechercher_meilleur_C(pipeline_m0, Xtr, ytr, GRILLE_C, seed=SEED)
p0 = modele_m0.predict_proba(Xte)[:, 1]
_ = evaluer("M0", yte, p0)

num_m1 = DECLARATIF_NUM + COMPORTEMENTAL_NUM
pipeline_m1 = construire_pipeline(num_m1, DECLARATIF_CAT, seed=SEED)
modele_m1, _ = rechercher_meilleur_C(pipeline_m1, Xtr, ytr, GRILLE_C, seed=SEED)
p1 = modele_m1.predict_proba(Xte)[:, 1]
_ = evaluer("M1", yte, p1)

M0   | AUC 0.653 | Gini 0.307 | KS 0.268


M1   | AUC 0.666 | Gini 0.333 | KS 0.275


## Étape 6 — Palier M2 : construire et évaluer

M1 **+** les parts de catégories (`PART_MOMO`, `PART_AGIOS`, `PART_NJANGUI`, ...). Même logique de recherche de `C` que pour M0/M1.

In [9]:
num_m2 = num_m1 + PART_COLS

pipeline_m2 = construire_pipeline(num_m2, DECLARATIF_CAT, seed=SEED)
modele_m2, cv_m2 = rechercher_meilleur_C(pipeline_m2, Xtr, ytr, GRILLE_C, seed=SEED)

print(f"meilleur C retenu pour M2 : {modele_m2.named_steps['clf'].C}\n")
p2 = modele_m2.predict_proba(Xte)[:, 1]
_ = evaluer("M2", yte, p2)

coefs_m2 = extraire_coefficients(modele_m2, num_m2, DECLARATIF_CAT)
coefs_m2.head(15)

meilleur C retenu pour M2 : 0.01

M2   | AUC 0.719 | Gini 0.438 | KS 0.350


,variable,coefficient
0,secteur_formel,-0.258254
1,secteur_informel,0.258131
2,PART_AGIOS,0.166204
3,tx_par_jour,-0.161487
4,PART_NJANGUI,-0.158089
5,PART_LOYER,0.149047
6,revenu_declare,-0.118990
7,nb_jours_actifs,-0.115899
8,outflow,-0.111090
9,cv_abs,0.104265


**Lecture.** Contrairement aux composantes SVD de M3 (plus bas), chaque `PART_*` est directement interprétable : un coefficient positif sur `PART_AGIOS` ou `PART_PARI` (agios/découvert, paris) signale un lien direct avec le risque ; un coefficient négatif sur `PART_NJANGUI` (tontine) est cohérent avec un filet de sécurité social qui réduit le risque. C'est ce niveau de détail — la *catégorie* de chaque transaction — que M1 ne pouvait pas voir, et c'est lui qui explique le saut d'AUC de M1 à M2 (test de DeLong plus bas). Comme pour M0 vs M1 dans `baseline.ipynb`, gardez à l'esprit que M1 et M2 n'ont pas forcément retenu le même `C` : une comparaison de magnitude brute entre leurs deux tables de coefficients resterait sujette au même piège.

### Référence non-linéaire à M2 — XGBoost et Random Forest

M2 est jusqu'ici le meilleur palier (AUC 0,719, régression logistique). Une régression logistique reste un modèle **linéaire** dans l'espace des variables transformées : elle peut sous-estimer un signal si la relation entre une variable et le risque est non monotone, ou si des variables interagissent (ex. `PART_AGIOS` × `secteur`). On entraîne ici deux modèles non linéaires de référence, sur **exactement le même split train/test et les mêmes variables que M2** (`num_m2 + DECLARATIF_CAT`, `Xtr`/`Xte`/`ytr`/`yte` déjà construits plus haut) :

- **XGBoost** — `scale_pos_weight` = ratio négatifs/positifs du train, l'équivalent pour un modèle à base d'arbres du `class_weight="balanced"` de la régression logistique ;
- **Random Forest** — `class_weight="balanced"` nativement.

Le code de ces deux pipelines vit dans `construire_pipeline_arbre` (`scoring_utils.py`) : même prétraitement (`StandardScaler` + `OneHotEncoder`) que `construire_pipeline`, seule l'étape finale change. Aucune recherche d'hyperparamètres n'est faite ici (contrairement à la recherche de `C` des paliers linéaires) — l'objectif est une comparaison de référence à budget de réglage raisonnable, pas un modèle optimisé.

Comparaison à la régression logistique M2 par **deux méthodes indépendantes** :
1. le test de DeLong déjà utilisé plus haut (`delong_test`, tient compte de la corrélation entre modèles évalués sur le même test) ;
2. un **intervalle de confiance bootstrap simple** sur l'AUC brute de chaque modèle (`bootstrap_auc_ci`, rééchantillonnage non paramétrique du test, sans hypothèse de corrélation) — une vérification indépendante, demandée en complément de DeLong.

In [10]:
from scoring_utils import bootstrap_auc_ci, construire_pipeline_arbre

pipeline_m2_xgb = construire_pipeline_arbre(num_m2, DECLARATIF_CAT, y=ytr, modele="xgboost", seed=SEED)
pipeline_m2_xgb.fit(Xtr, ytr)
p2_xgb = pipeline_m2_xgb.predict_proba(Xte)[:, 1]
resultat_m2_xgb = evaluer("M2-XGB", yte, p2_xgb)

pipeline_m2_rf = construire_pipeline_arbre(num_m2, DECLARATIF_CAT, modele="random_forest", seed=SEED)
pipeline_m2_rf.fit(Xtr, ytr)
p2_rf = pipeline_m2_rf.predict_proba(Xte)[:, 1]
resultat_m2_rf = evaluer("M2-RF", yte, p2_rf)

M2-XGB | AUC 0.722 | Gini 0.445 | KS 0.367


M2-RF | AUC 0.700 | Gini 0.399 | KS 0.370


In [11]:
print("Comparaison à M2 (régression logistique, AUC référence) — test de DeLong :\n")
for nom, p_autre in [("M2-XGB", p2_xgb), ("M2-RF", p2_rf)]:
    aucs, pval = delong_test(yte, p2, p_autre)
    conclusion = "significatif" if pval < 0.05 else "non significatif"
    print(f"DeLong M2(logit)->{nom} : AUC {aucs[0]:.3f} -> {aucs[1]:.3f} | p = {pval:.2e} ({conclusion})")

print("\nIC bootstrap (2000 réplications, jeu de test) sur l'AUC brute de chaque modèle :")
for nom, p_mod in [("M2 (logit)", p2), ("M2-XGB", p2_xgb), ("M2-RF", p2_rf)]:
    auc_obs, (lo, hi) = bootstrap_auc_ci(yte, p_mod, seed=SEED)
    print(f"  {nom:12} AUC {auc_obs:.3f}  IC95% [{lo:.3f} ; {hi:.3f}]")

Comparaison à M2 (régression logistique, AUC référence) — test de DeLong :

DeLong M2(logit)->M2-XGB : AUC 0.719 -> 0.722 | p = 9.00e-01 (non significatif)
DeLong M2(logit)->M2-RF : AUC 0.719 -> 0.700 | p = 3.39e-01 (non significatif)

IC bootstrap (2000 réplications, jeu de test) sur l'AUC brute de chaque modèle :


  M2 (logit)   AUC 0.719  IC95% [0.654 ; 0.778]


  M2-XGB       AUC 0.722  IC95% [0.656 ; 0.785]


  M2-RF        AUC 0.700  IC95% [0.633 ; 0.763]


**Lecture.** Sur ce jeu de test (400 clients, ~63 défauts) :

| Modèle | AUC | IC95% bootstrap |
|---|---|---|
| M2 (régression logistique) | 0,719 | [0,654 ; 0,778] |
| M2-XGB (XGBoost) | 0,722 | [0,656 ; 0,785] |
| M2-RF (Random Forest) | 0,700 | [0,633 ; 0,763] |

XGBoost affiche une AUC brute très légèrement supérieure (+0,003 point face à la régression logistique), Random Forest est en retrait (-0,019 point). Mais **aucun des deux écarts n'est statistiquement significatif** : DeLong donne p = 0,90 (logit → XGB) et p = 0,34 (logit → RF), et les trois IC bootstrap se recouvrent très largement. Avec ~63 défauts en test, le test manque de puissance pour trancher un écart de cette taille — un résultat « non significatif » ne veut pas dire « pas d'écart réel », seulement que celui-ci n'est pas mesurable avec confiance à cet effectif.

**Conclusion.** Sur ce jeu de données, la non-linéarité (arbres) n'apporte pas de gain démontré par rapport à la régression logistique au palier M2 : le signal capté par `PART_*` et les variables comportementales semble à dominante additive/monotone, que la régression logistique linéarise déjà bien via le prétraitement (standardisation, one-hot). XGBoost reste une référence utile à garder pour une ré-évaluation si le jeu de données réel (volumes plus grands, relations plus complexes) s'y prête mieux — mais ne remplace pas la régression logistique ici, qui garde l'avantage de l'interprétabilité directe (table de coefficients) pour un usage réglementaire.

### Bonus — SMOTE vs `class_weight="balanced"` (M2)

M2 gère le déséquilibre de classes par pondération (`class_weight="balanced"`). Alternative classique : sur-échantillonner synthétiquement la classe minoritaire (SMOTE, Chawla et al. 2002) *avant* l'entraînement plutôt que de pondérer les erreurs. Comparaison à `C` identique (celui déjà retenu pour M2) pour isoler l'effet du seul traitement du déséquilibre — SMOTE est appliqué **uniquement sur le train**, à l'intérieur d'un pipeline `imblearn` (jamais sur le test, sinon fuite par duplication de voisins synthétiques).

In [12]:
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as PipelineImb
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import OneHotEncoder, StandardScaler

C_m2 = modele_m2.named_steps["clf"].C  # même C que M2 (class_weight="balanced"), pour isoler l'effet SMOTE vs class_weight

pretraitement_m2 = ColumnTransformer([
    ("num", StandardScaler(), num_m2),
    ("cat", OneHotEncoder(handle_unknown="ignore"), DECLARATIF_CAT),
])
pipeline_m2_smote = PipelineImb([
    ("pre", pretraitement_m2),
    ("smote", SMOTE(random_state=SEED)),
    ("clf", LogisticRegression(C=C_m2, class_weight=None, max_iter=2000, random_state=SEED)),
])
pipeline_m2_smote.fit(Xtr, ytr)
p2_smote = pipeline_m2_smote.predict_proba(Xte)[:, 1]
resultat_m2_smote = evaluer("M2-SMOTE", yte, p2_smote)

aucs, pval = delong_test(yte, p2, p2_smote)
conclusion = "significatif" if pval < 0.05 else "non significatif"
print(f"\nDeLong M2(class_weight=balanced)->M2(SMOTE) : AUC {aucs[0]:.3f} -> {aucs[1]:.3f} | p = {pval:.2e} ({conclusion})")

M2-SMOTE | AUC 0.716 | Gini 0.431 | KS 0.348

DeLong M2(class_weight=balanced)->M2(SMOTE) : AUC 0.719 -> 0.716 | p = 7.28e-01 (non significatif)


**Lecture.** M2-SMOTE (AUC 0,716) et M2 avec `class_weight="balanced"` (AUC 0,719) sont quasiment identiques (DeLong p = 0,73, non significatif) : sur ce jeu de données, les deux traitements du déséquilibre de classes se valent. C'est cohérent avec la littérature (Chawla et al. 2002 vs pondération des erreurs sont deux façons différentes d'atteindre le même objectif — corriger le biais du modèle vers la classe majoritaire — sans qu'aucune ne domine systématiquement l'autre) : `class_weight="balanced"` reste préférable ici par simplicité (pas d'hyperparamètre de sur-échantillonnage à régler, pas de risque de sur-ajustement aux voisins synthétiques) à résultat égal.

## Étape 7 — Palier M3 : construire et évaluer

M2 **+** la représentation vectorielle du texte (`texte_col="texte"` dans `construire_pipeline`, voir Étape 3).

In [13]:
pipeline_m3 = construire_pipeline(num_m2, DECLARATIF_CAT, texte_col="texte", seed=SEED)
modele_m3, cv_m3 = rechercher_meilleur_C(pipeline_m3, Xtr, ytr, GRILLE_C, seed=SEED)

print(f"meilleur C retenu pour M3 : {modele_m3.named_steps['clf'].C}\n")
p3 = modele_m3.predict_proba(Xte)[:, 1]
_ = evaluer("M3", yte, p3)

coefs_m3 = extraire_coefficients(modele_m3, num_m2, DECLARATIF_CAT)
coefs_m3.head(15)

meilleur C retenu pour M3 : 1



M3   | AUC 0.718 | Gini 0.437 | KS 0.327


,variable,coefficient
0,texte_svd_1,2.770702
1,texte_svd_24,-1.632406
2,texte_svd_14,-1.631105
3,texte_svd_11,1.339194
4,texte_svd_22,-1.287823
5,texte_svd_3,-1.195894
6,texte_svd_9,-1.145170
7,texte_svd_21,1.074089
8,texte_svd_27,1.026443
9,texte_svd_19,0.916057


In [14]:
# les composantes texte_svd_0..29 ne se lisent pas mot à mot (contrairement aux PART_*) ;
# on regarde seulement leur poids agrégé face au reste du modèle
est_texte = coefs_m3.variable.str.startswith("texte_svd_")
part_poids_texte = coefs_m3.loc[est_texte, "coefficient"].abs().sum() / coefs_m3.coefficient.abs().sum()

print(f"part du poids total (|coefficient|) porté par les 30 composantes texte : {part_poids_texte:.1%}")

part du poids total (|coefficient|) porté par les 30 composantes texte : 79.3%


**Lecture — attention à ce chiffre, il est instable.** La part de poids portée par les composantes texte est passée de 15,7 % à 79,3 % après la correction d'un bug de `categorisation.py` (le regex qui isolait les codes d'agence `AG...` supprimait aussi le mot « AGIOS » lui-même, voir `tests/test_categorisation.py` — corrigé, précision de règles 0,710→0,721). En creusant la grille de `C` retenue pour M3 (non affichée par cette cellule mais recalculée à part) : les AUC de validation croisée pour `C ∈ {0,001 ... 100}` sont toutes comprises entre 0,695 et 0,700, à moins de 0,001 les unes des autres — un **quasi-ex-aequo statistique**. `GridSearchCV` a basculé du côté `C=0,01` vers `C=1` suite à cette correction, une différence de régularisation qui change fortement l'échelle des coefficients (et donc leur part de poids relative) **sans que cela corresponde à un vrai changement de signal**. Conclusion méthodologique à retenir : cette mesure de « part de poids » est fragile dès que plusieurs valeurs de `C` sont statistiquement indiscernables en validation croisée — le test de DeLong ci-dessous, sur l'AUC réellement mesurée en test, reste la mesure qui tranche.

## Étape 8 — La chaîne de comparaisons DeLong

M1 → M2 → M3, chacun comparé au précédent, sur les mêmes clients de test.

In [15]:
chaine_delong(yte, {"M1": p1, "M2": p2, "M3": p3})

DeLong M1->M2 : AUC 0.666 -> 0.719 | p = 2.97e-03 (significatif)
DeLong M2->M3 : AUC 0.719 -> 0.718 | p = 9.69e-01 (non significatif)


## Étape 9 — Audit d'équité (modèle M3)

Au seuil d'approbation médian de M3 : ratio d'approbation par secteur et par sexe (règle des 4/5), et — grâce à `gt_defaut_true` (vérité-terrain, jamais utilisée comme variable) — la part des **vrais bons** refusés par groupe. C'est cette dernière mesure qui distingue un écart de risque réel d'une injustice injectée par le biais d'étiquette. La correction de ce biais est traitée dans `equite.ipynb`, pas ici.

In [16]:
for groupe in ["secteur", "sexe"]:
    audit_equite(dfte, p3, groupe, gt_true_col="gt_defaut_true")
    print()

taux d'approbation par secteur : {'formel': 0.776, 'informel': 0.267}
ratio (règle des 4/5) : 0.34 -> disparate impact
refus des vrais bons par secteur : {'formel': 0.183, 'informel': 0.71}

taux d'approbation par sexe : {'F': 0.517, 'M': 0.481}
ratio (règle des 4/5) : 0.93 -> pas de signal au seuil des 4/5
refus des vrais bons par sexe : {'F': 0.457, 'M': 0.469}



## Lecture des résultats

**Le texte fait la différence, mais par les règles.** M1→M2 est un gain significatif au test de DeLong (p = 2,97e-03) : dès qu'on connaît la *part de chaque catégorie* (dash, paris, agios pour la tension ; njangui pour le filet social, invisible ailleurs), l'AUC monte nettement (0,666 → 0,719). C'est le moment où comprendre le texte « paie ».

**Les embeddings n'ajoutent pas grand-chose ici (M2→M3), malgré une table de coefficients trompeuse.** Le test de DeLong reste la mesure qui tranche : M2→M3 est non significatif (0,719 → 0,718, p = 0,97) — l'AUC de test ne bouge pas. La part de poids des composantes texte dans les coefficients (79,3 %, Étape 7 ci-dessus) pourrait laisser croire l'inverse, mais elle s'est révélée instable (elle dépend d'un choix de `C` presque à égalité en validation croisée, voir la lecture juste au-dessus) — un rappel qu'une table de coefficients ne remplace pas un test statistique sur la métrique qui compte réellement. Sur ce jeu au lexique connu, les règles captent déjà l'essentiel du signal ; le TF-IDF caractères n'apporte pas de gain mesurable. La valeur des embeddings apparaîtrait sur un vocabulaire ouvert / non anticipé, que le lexique manuel ne couvrirait pas — à documenter comme limite de ce jeu de données, pas à surjouer.

**Ni les règles ni les embeddings ne voient le mensonge.** La précision de catégorisation chute sur les lignes trompeuses (voir `precision` vs `precision_hors_mensonge`, Étape 2) : un pari libellé « ACHAT DIVERS » reste invisible aux deux approches. C'est ce que le raisonnement d'un LLM sur le *contexte* (montants, régularité) devra tenter au palier M5 — et peut-être échouer aussi, ce qu'il faudra mesurer honnêtement.

**Équité — le point à ne jamais lâcher.** L'audit ci-dessus mesure deux choses différentes : le ratio d'approbation (peut refléter un vrai écart de risque) et le refus des **vrais bons** par groupe (l'injustice injectée par le biais d'étiquette). Par secteur, un écart marqué sur les deux mesures ; par sexe, généralement plus équilibré. La correction de ce biais secteur (seuils différenciés par groupe, repondération des données d'entraînement) est traitée dans [`equite.ipynb`](equite.ipynb) — voir aussi la mise en garde ajoutée dans ce dernier notebook sur la part de ce biais qui est construite par le générateur de données, et pas seulement une découverte empirique.